# ฝึกโมเดลพื้นที่ผ่านได้จาก CVAT — เริ่มที่นี่

ใช้ไฟล์ **part1.zip** จาก CVAT (Segmentation mask แบบไม่รวมภาพ) และ **part1.mov ต้นฉบับ** ที่ upload เข้า CVAT

1. เลือก **Runtime → Change runtime type → T4 GPU** ถ้ามีให้ใช้ แล้วกด Save
2. กด ▶ ของช่องโค้ดตามลำดับ 1–5 ด้านล่าง รอแต่ละช่องเสร็จก่อนกดช่องถัดไป
3. ช่อง 2 จะให้เลือกไฟล์สองไฟล์ ช่อง 5 จะดาวน์โหลดโมเดลที่ฝึกแล้ว

Colab มีโหมดใช้ฟรี แต่ไม่ได้รับประกัน GPU ว่าง ไม่ต้องสมัคร Pro เพื่อเปิด notebook นี้ ถ้าได้ CPU โค้ดจะทดลองฝึก 3 epochs และอาจช้า; GPU ฝึกต้นแบบ 20 epochs

นี่เป็นการฝึกต้นแบบจากข้อมูลการบันทึกเดียว ไม่ใช่การวัดความแม่นยำบนพื้นที่ใหม่ ค่า loss และภาพตัวอย่างหลังฝึกมาจากชุดฝึก ไม่ใช่ชุดทดสอบ


## 1 — ติดตั้งเครื่องมือ

ช่องนี้โหลดโค้ดจาก GitHub และติดตั้ง dependency ใน Colab ไม่เปลี่ยน venv บน Ubuntu ของคุณ


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/husk')
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'work',
                    'https://github.com/jorjor140900/husk.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO / 'requirements-segmentation.txt')], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import torch
print('พร้อมใช้งาน | GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ไม่มี — ใช้ CPU ได้แต่ช้ากว่า')


## 2 — อัปโหลด part1.zip และ part1.mov

กดปุ่ม Choose Files แล้วเลือกสองไฟล์พร้อมกัน (Ctrl+คลิก) ใช้ ZIP annotation และวิดีโอต้นฉบับ ไม่ใช้ ZIP ชุดภาพที่ Codex เคยสร้างให้


In [ ]:
from datetime import datetime, timezone
from google.colab import files

STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
INPUTS = Path('/content/husky-inputs') / STAMP
INPUTS.mkdir(parents=True)
os.chdir(INPUTS)
try:
    uploaded = files.upload()
finally:
    os.chdir(REPO)
paths = [INPUTS / name for name in uploaded]
archives = [p for p in paths if p.suffix.lower() == '.zip']
videos = [p for p in paths if p.suffix.lower() in ('.mov', '.mp4', '.avi')]
if len(archives) != 1 or len(videos) != 1:
    raise ValueError('กรุณารันช่องนี้ใหม่ แล้วเลือก ZIP annotation 1 ไฟล์และวิดีโอต้นฉบับ 1 ไฟล์พร้อมกัน')
ANNOTATIONS, VIDEO = archives[0], videos[0]
del uploaded
print('ZIP:', ANNOTATIONS.name, '| วิดีโอ:', VIDEO.name)


## 3 — จับคู่ภาพกับคำตอบ และตรวจภาพรวม

สีเขียว=ผ่านได้ สีแดง=ห้ามผ่าน ส่วนมืด=ไม่ label ซึ่งจะไม่ใช้คำนวณ loss
ตรวจว่าขอบ label ตรงกับเสาและพื้นจริงก่อนกดช่องฝึก หากไม่ตรงให้ตรวจวิดีโอและเลขเฟรมก่อน


In [ ]:
from IPython.display import Image, display
from tools.prepare_cvat_dataset import prepare_dataset

RUN = Path('/content/husky-runs') / STAMP
DATASET = RUN / 'dataset'
summary = prepare_dataset(ANNOTATIONS, VIDEO, DATASET, 'front-building-session-01')
print('เตรียมได้', summary['samples'], 'คู่ภาพ/คำตอบ')
print('เลขเฟรม:', summary['source_frame_indices_zero_based'])
display(Image(filename=str(DATASET / 'contact-sheet.png'), width=1100))


## 4 — เริ่มฝึก SegFormer-B0

โหลด pretrained model สาธารณะจาก Hugging Face แล้ว fine-tune กับคำตอบของคุณ ไม่มีการฝึกจากน้ำหนักสุ่มแทนโดยเงียบ ๆ หากดาวน์โหลดไม่สำเร็จ ช่องนี้จะแจ้งข้อผิดพลาด

รอจนเห็น **Model saved to ...** โมเดลจะบันทึก checkpoint เมื่อจบแต่ละ epoch ตัวเลข training loss ไม่ใช่ accuracy


In [ ]:
TRAIN_STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
MODEL = RUN / ('train-' + TRAIN_STAMP) / 'husky-segformer'
EPOCHS = 20 if torch.cuda.is_available() else 3
HEIGHT, WIDTH = (384, 640) if torch.cuda.is_available() else (256, 448)
print('เริ่มฝึก', EPOCHS, 'epochs บน', 'GPU' if torch.cuda.is_available() else 'CPU')
subprocess.run([
    sys.executable, '-m', 'tools.train_segmentation',
    '--dataset', str(DATASET), '--output', str(MODEL),
    '--epochs', str(EPOCHS), '--batch-size', '2', '--device', 'auto',
    '--height', str(HEIGHT), '--width', str(WIDTH),
], cwd=REPO, check=True)
for example in sorted((MODEL / 'training_previews').glob('*.jpg')):
    display(Image(filename=str(example), width=1100))
print('ตัวอย่างข้างบนมาจากชุดฝึก ยังไม่ใช่ผลทดสอบอิสระ')


## 5 — ดาวน์โหลดโมเดลกลับเครื่อง

ดาวน์โหลดจาก Colab โดยตรง ไม่ต้องดาวน์โหลด artifact จาก Codex เก็บ ZIP นี้ไว้ เพราะไฟล์ใน runtime ของ Colab อาจหายเมื่อ session สิ้นสุด


In [ ]:
import shutil

for name in ('config.json', 'preprocessor_config.json', 'model.safetensors', 'training_report.json'):
    if not (MODEL / name).is_file():
        raise FileNotFoundError('ยังไม่มีโมเดลครบชุด: ' + name + ' — ตรวจช่องฝึกก่อน')
archive = shutil.make_archive(str(RUN / 'husky-segformer'), 'zip',
                              root_dir=MODEL.parent, base_dir=MODEL.name)
print('ดาวน์โหลดโมเดล:', Path(archive).name)
files.download(archive)


## ใช้โมเดลกับ webcam บน Ubuntu หลังดาวน์โหลดเสร็จ

เปิด Terminal บน Ubuntu ใน repository ที่เคยเปิดกล้องได้ คำสั่งติดตั้งด้านล่างใช้ PyTorch CPU และคง OpenCV แบบ GUI ที่คุณใช้อยู่ ใช้โฟลเดอร์โมเดลใหม่เมื่อเปลี่ยนรุ่น

```bash
cd ~/husk-live
git pull --ff-only
source .venv/bin/activate
python -m pip install --index-url https://download.pytorch.org/whl/cpu 'torch==2.6.0'
python -m pip install 'transformers==4.49.0' 'safetensors>=0.4.3,<1' 'Pillow>=10,<12'
mkdir -p models
unzip ~/Downloads/husky-segformer.zip -d models
python RoadDetection.py --camera 0 --show --config config.example.json --model models/husky-segformer --device cpu
```

ถ้า browser บันทึก ZIP ไว้ตำแหน่งอื่น ให้แก้ path ของ unzip ให้ตรง **เก็บทั้งโฟลเดอร์โมเดล** รวม config และ preprocessor ไม่ใช้เฉพาะไฟล์ weights

โค้ดใช้ mask จากโมเดลแทน color/Canny แล้วใช้ตัวสร้าง reference เดิม ยังไม่มี ROS หรือคำสั่งควบคุมหุ่น ข้อมูล part1 ชุดเดียวใช้ทดลองกระบวนการได้ แต่ต้องเพิ่มภาพ/การบันทึกอิสระก่อนสรุปว่าแยกเสา เงา หรือพรมได้แม่น
